# Qwen3.5-4B: LoRA training


In [ ]:
%pip install -q "torch==2.8.0" "torchvision==0.23.0" "torchaudio==2.8.0" "xformers==0.0.32.post2" "unsloth==2026.11.1" "unsloth_zoo==2026.11.1" "transformers==5.2.0" "trl==0.22.2" "torchao==0.16.0"

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/data/prepared/plant_care_train_1000")
RUN_DIR = Path("/content/data/runs/plant_care_train_1000/qwen35_4b_lora")
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers.trainer_utils import get_last_checkpoint

MODEL_ID = "Qwen/Qwen3.5-4B"
REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    revision=REVISION,
    use_exact_model_name=True,
    max_seq_length=2048,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    text_only=True,
    use_gradient_checkpointing="unsloth",
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.peft_config["default"].revision = REVISION

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:487: UserWarning: Unsloth: torchcodec is installed but cannot load its native libraries although FFmpeg is on the loader path; likely an FFmpeg major it does not support (it takes 4 to 8), a missing CUDA NPP runtime (nvidia-npp), or a build that does not match this torch; audio datasets decode through soundfile and PyAV instead (wav/flac/mp3/ogg, m4a/aac/webm). torchcodec 0.11.0+cu130 is incompatible with torch 2.8.0+cu128; install a matching build with `pip install --index-url https://download.pytorch.org/whl/cu128 'torchcodec>=0.7,<0.8.0'`.
  disable_torchcodec_if_broken()


🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.13/dist-packages/unsloth/models/rl.py:1580: UserWarning: Torchaudio's I/O functions now support per-call backend dispatch. Importing backend implementation directly is no longer guaranteed to work. Please use `backend` keyword with load/save/info function, instead of calling the underlying implementation directly.
  if getattr(module, "prepare_peft_model", None) is original:


==((====))==  Unsloth 2026.11.1: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [ ]:
dataset = load_dataset("json", data_files={
    "train": str(DATA_DIR / "train.jsonl"),
    "validation": str(DATA_DIR / "validation.jsonl"),
})

assert len(dataset["train"]) == 800
assert len(dataset["validation"]) == 200

def format_example(row):
    messages = row["messages"]
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    assert text.startswith(prompt)
    return {"prompt": prompt, "completion": text[len(prompt):]}

dataset = dataset.map(format_example, remove_columns=dataset["train"].column_names)
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        output_dir=str(RUN_DIR / "checkpoints"),
        num_train_epochs=1,
        max_length=2048,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=1,
        eval_accumulation_steps=1,
        learning_rate=2e-4,
        warmup_steps=10,
        weight_decay=0.001,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        fp16=False,
        bf16=False,
        completion_only_loss=True,
        packing=False,
        dataset_num_proc=1,
        logging_steps=1,
        logging_first_step=True,
        disable_tqdm=True,
        report_to="none",
        eval_on_start=True,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=2,
        seed=3407,
        data_seed=3407,
    ),
)
(RUN_DIR / "training_config.json").write_text(trainer.args.to_json_string())

Unsloth: Switching to float32 training since model cannot work with float16


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
result = trainer.train(resume_from_checkpoint=get_last_checkpoint(trainer.args.output_dir))

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 21,233,664 of 4,226,984,960 (0.50% trained)


{'eval_loss': '2.43', 'eval_runtime': '92.14', 'eval_samples_per_second': '2.17', 'eval_steps_per_second': '2.17', 'epoch': 0}


{'loss': '2.32', 'grad_norm': '1.022', 'learning_rate': '0', 'epoch': '0.005'}


{'loss': '2.387', 'grad_norm': '1.026', 'learning_rate': '2e-05', 'epoch': '0.01'}
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


{'loss': '2.357', 'grad_norm': '0.8814', 'learning_rate': '4e-05', 'epoch': '0.015'}


... (210 outputs left out)

{'loss': '1.995', 'grad_norm': '1.511', 'learning_rate': '1.053e-06', 'epoch': '1'}


{'eval_loss': '1.842', 'eval_runtime': '76.9', 'eval_samples_per_second': '2.601', 'eval_steps_per_second': '2.601', 'epoch': '1'}


Unsloth: Restored added_tokens_decoder metadata in /content/data/runs/plant_care_train_1000/qwen35_4b_lora/checkpoints/checkpoint-200/tokenizer_config.json.


{'train_runtime': '1925', 'train_samples_per_second': '0.416', 'train_steps_per_second': '0.104', 'train_loss': '1.903', 'epoch': '1'}


In [ ]:
assert trainer.state.global_step == 200
trainer.save_model(str(RUN_DIR / "adapter"))
tokenizer.save_pretrained(str(RUN_DIR / "adapter"))
trainer.save_state()
trainer.save_metrics("train", result.metrics)
print(RUN_DIR / "adapter")
result.metrics

Unsloth: Restored added_tokens_decoder metadata in /content/data/runs/plant_care_train_1000/qwen35_4b_lora/adapter/tokenizer_config.json.


/content/data/runs/plant_care_train_1000/qwen35_4b_lora/adapter


{'train_runtime': 1924.8355,
 'train_samples_per_second': 0.416,
 'train_steps_per_second': 0.104,
 'train_loss': 1.9027829194068908,
 'epoch': 1.0}